<div dir="rtl" style="text-align:right"><h1>שיעור 3, חלק 2: איך מודל שפה כותב – שלב B: המודל בוחר את ההמשך</h1>
<p><b>המחברת הזו היא לשלב B («המודל בוחר את ההמשך») בשיעור באתר הקורס – ורק לו.</b> את ההסבר קוראים באתר; כאן מריצים ומשנים.</p>
<p>מריצים תא אחרי תא בלחיצה על ▶, מלמעלה למטה. מעבד רגיל (<code>CPU</code>) מספיק; אין צורך ב-<code>GPU</code>, במפתח <code>API</code> או במנוי. <b>אין חובת הגשת מחברת.</b> תא ההכנה מוריד מודל פתוח של כ-1GB; ההורדה לוקחת בדרך כלל דקה או שתיים. אם מופיעה אזהרה על <code>HF_TOKEN</code> – אפשר להתעלם ממנה: המודל ציבורי.</p></div>

In [1]:
# Setup: transformers and torch are already installed on Google Colab.
# Loads the small open model Qwen/Qwen2.5-0.5B (about 1GB; usually a minute or two).
import math

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# "eager" attention lets the model return its attention weights; the predictions are the same.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32, attn_implementation="eager")
model.eval()
print(f"Loaded {MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} billion parameters")


def next_token_table(context, k=8):
    """The k most likely next tokens after `context`, with probabilities."""
    ids = tokenizer(context, return_tensors="pt").input_ids
    with torch.no_grad():
        probs = torch.softmax(model(ids).logits[0, -1], dim=-1)
    values, indices = probs.topk(k)
    print("Context:", context)
    for p, i in zip(values.tolist(), indices.tolist()):
        piece = tokenizer.decode([i]).replace(" ", "·").replace("\n", "\\n")
        print(f"  {piece:>12}  {p:6.1%}  " + "█" * round(p * 40))
    print()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:03<17:50,  3.70s/it]

Loading weights:  80%|███████▉  | 231/290 [00:03<00:00, 84.45it/s]

Loading weights: 100%|██████████| 290/290 [00:04<00:00, 65.72it/s]

Loaded Qwen/Qwen2.5-0.5B: 0.49 billion parameters


<div dir="rtl" style="text-align:right"><h2>2 | חיזוי הטוקן הבא</h2>
<p>מודל שפה מקבל את רצף הטוקנים – <b>ההקשר</b> – ומחשב לכל טוקן אפשרי באוצר שלו הסתברות להיות הטוקן הבא. אחר כך נבחר טוקן אחד, מצורף לרצף, והחישוב חוזר. כך נבנית תשובה שלמה, טוקן אחרי טוקן.</p>
<p>נשתמש במודל הפתוח <code>Qwen/Qwen2.5-0.5B</code>: כחצי מיליארד פרמטרים, גרסת בסיס (לא צ׳טבוט), שרץ על מעבד רגיל. <b>ההגבלות, ביושר:</b></p>
<ul>
<li>זה <b>לא</b> המודל של הצ׳טבוט שלכם. המודלים שמאחורי צ׳טבוטים מוכרים גדולים ממנו בדרך כלל בהרבה (החברות לא תמיד מפרסמות כמה), ועברו אימון נוסף לשיחה.</li>
<li>הוא חלש בעברית בהרבה מאשר באנגלית. לכן נריץ כל דוגמה בשתי השפות.</li>
<li>המספרים מראים מה <b>המודל הזה</b> מחשב. הם לא מודדים מה קורה בתוך הצ׳טבוט.</li>
</ul>
<p><b>זוכרים מחלק 1:</b> פרמטרים הם המספרים שהאימון קבע. במהלך החיזוי <b>הפרמטרים של המודל לא משתנים</b>. מה שמשתנה בין דוגמה לדוגמה הוא רק ההקשר.</p></div>

<div dir="rtl" style="text-align:right"><h3>2א | אותה התחלה, הקשר אחר</h3>
<p>שני משפטים באנגלית שמסתיימים כמעט באותו אופן. רק תחילת המשפט שונה. הסתכלו על הטוקנים המובילים בכל אחד.</p><blockquote>① נחשו לפני ההרצה: איזה טוקן יוביל בכל משפט?</blockquote></div>

In [2]:
next_token_table("Every morning before work I drink a cup of")
next_token_table("At the festive dinner we drank a glass of")

Context: Every morning before work I drink a cup of
       ·coffee   45.5%  ██████████████████
          ·tea   15.4%  ██████
          ·hot    6.8%  ███
        ·green    5.8%  ██
       ·herbal    3.4%  █
             ·    1.8%  █
        ·black    1.4%  █
        ·water    1.1%  

Context: At the festive dinner we drank a glass of
         ·wine   18.0%  ███████
    ·champagne    6.6%  ███
         ·beer    3.8%  ██
       ·orange    3.4%  █
        ·brand    2.6%  █
         ·milk    2.5%  █
          ·rum    2.3%  █
        ·lemon    2.3%  █



<div dir="rtl" style="text-align:right"><h3>2ב | ובעברית?</h3>
<p>אותה דוגמה בעברית. שימו לב: רבים מהטוקנים המובילים הם <b>חלקי מילים</b> – בדיוק מה שראינו בשלב A («המודל מקבל רצף יחידות»). כדי להשוות מילים שלמות, התא שאחריו מחשב לכל מילה מועמדת את ההסתברות של כל הטוקנים שלה ברצף, ומציג איזה <b>חלק</b> מקבלת כל מילה <b>מתוך המועמדות שבחרנו בלבד</b>. זה לא ההסתברות מכל ההמשכים האפשריים.</p></div>

In [3]:
next_token_table("בבוקר, לפני העבודה, אני שותה כוס")
next_token_table("בערב, עם הארוחה החגיגית, שתינו כוס")

Context: בבוקר, לפני העבודה, אני שותה כוס
             ה   15.8%  ██████
           ·של    5.5%  ██
            ·ח    3.5%  █
            ·ק    3.2%  █
            ות    3.1%  █
            ים    2.3%  █
            ·ש    2.2%  █
            ·ב    2.1%  █



Context: בערב, עם הארוחה החגיגית, שתינו כוס
             ה   11.9%  █████
            ות    9.8%  ████
            ים    7.4%  ███
           ·של    3.0%  █
             י    2.2%  █
            ·מ    1.4%  █
            ·ח    1.4%  █
            ·ו    1.3%  █



In [4]:
def compare_words(context, words):
    """Each candidate word's share among `words` only, from the model's
    probability of all its tokens after `context`."""
    context_ids = tokenizer(context).input_ids
    log_probs = {}
    for word in words:
        word_ids = tokenizer(" " + word).input_ids
        ids = torch.tensor([context_ids + word_ids])
        with torch.no_grad():
            lp = torch.log_softmax(model(ids).logits[0], dim=-1)
        log_probs[word] = sum(lp[len(context_ids) - 1 + j, t].item() for j, t in enumerate(word_ids))
    total = sum(math.exp(v) for v in log_probs.values())
    print("Context:", context)
    for word, v in sorted(log_probs.items(), key=lambda kv: -kv[1]):
        share = math.exp(v) / total
        print(f"  {word:>8}  {share:6.1%}  " + "█" * round(share * 40))
    print()


compare_words("Every morning before work I drink a cup of", ["coffee", "tea", "water", "wine"])
compare_words("At the festive dinner we drank a glass of", ["coffee", "tea", "water", "wine"])
compare_words("בבוקר, לפני העבודה, אני שותה כוס", ["קפה", "תה", "מים", "יין"])
compare_words("בערב, עם הארוחה החגיגית, שתינו כוס", ["קפה", "תה", "מים", "יין"])

Context: Every morning before work I drink a cup of
    coffee   73.3%  █████████████████████████████
       tea   24.9%  ██████████
     water    1.8%  █
      wine    0.0%  



Context: At the festive dinner we drank a glass of
      wine   89.1%  ████████████████████████████████████
       tea    5.2%  ██
     water    3.8%  ██
    coffee    1.9%  █



Context: בבוקר, לפני העבודה, אני שותה כוס
       מים   56.9%  ███████████████████████
       קפה   35.9%  ██████████████
       יין    6.5%  ███
        תה    0.7%  



Context: בערב, עם הארוחה החגיגית, שתינו כוס
       מים   75.6%  ██████████████████████████████
       קפה   14.0%  ██████
       יין    9.3%  ████
        תה    1.2%  



<div dir="rtl" style="text-align:right"><p><b>איך קוראים את התוצאות:</b> באנגלית ההקשר הזיז את ההסתברויות בבירור – משקה של בוקר מול משקה של ארוחה חגיגית. בעברית, בהרצה שלנו, ההזזה הייתה חלשה הרבה יותר, ו«מים» הוביל בשני ההקשרים. זו לא תקלה במחברת: זה מודל קטן שלמד מעט עברית. צ׳טבוט גדול יתנהג אחרת, ואת זה המחברת לא מודדת.</p>
<p>המספרים אצלכם עשויים להיות מעט שונים (גרסאות ספריות שונות), אבל הכיוון אמור להישמר.</p></div>

<div dir="rtl" style="text-align:right"><h3>2ג | התור שלכם</h3>
<p>③ שנו את ההקשר או את המילים המועמדות והריצו. נסו לשנות <b>פרט אחד בלבד</b> בהקשר בכל פעם, ובדקו מה זז.</p>
<blockquote>שאלות לחשיבה: מה השתנה בין שתי ההרצות – המודל או ההקשר? האם הסתברות גבוהה אומרת שההמשך <b>נכון</b>, או רק שהוא <b>סביר</b> לפי מה שהמודל למד?</blockquote></div>

In [5]:
compare_words("The bakery opens early, and every morning it sells fresh", ["bread", "cookies", "pizza", "ice cream"])  # <- change and run

# The same in Hebrew. The small model is much weaker in Hebrew, so expect odd results:
# compare_words("המאפייה נפתחת מוקדם, ובכל בוקר מוכרים בה", ["לחם", "עוגיות", "פיצה", "גלידה"])

Context: The bakery opens early, and every morning it sells fresh
     bread   99.3%  ████████████████████████████████████████
   cookies    0.7%  
     pizza    0.0%  
  ice cream    0.0%  



<div dir="rtl" style="text-align:right"><h3>2ד | טמפרטורה: אותו הקשר, הגרלות שונות</h3>
<p>המודל לא חייב לבחור תמיד את הטוקן הסביר ביותר: הוא <b>מגריל</b> לפי ההסתברויות. <b>טמפרטורה</b> משנה את ההגרלה: נמוכה – כמעט תמיד הטוקן הסביר ביותר; גבוהה – גם טוקנים פחות סבירים נבחרים לפעמים.</p>
<blockquote>① נחשו: בטמפרטורה 0.3, שלוש ההגרלות ייצאו דומות או שונות? ובטמפרטורה 1.5?</blockquote></div>

In [6]:
def sample(context, temperature, n=3, new_tokens=6):
    ids = tokenizer(context, return_tensors="pt").input_ids
    for seed in range(n):
        torch.manual_seed(seed)
        out = model.generate(ids, max_new_tokens=new_tokens, do_sample=True, temperature=temperature,
                             top_k=0, pad_token_id=tokenizer.eos_token_id)
        print(f"  T={temperature}: {context}" + tokenizer.decode(out[0, ids.shape[1]:]).split("\n")[0])


context = "Every morning before work I drink a cup of"  # <- ③ change one word, or try Hebrew
for temperature in (0.3, 1.0, 1.5):
    sample(context, temperature)

  T=0.3: Every morning before work I drink a cup of coffee. I like to have


  T=0.3: Every morning before work I drink a cup of coffee. I have a preference


  T=0.3: Every morning before work I drink a cup of coffee. The coffee is prepared


  T=1.0: Every morning before work I drink a cup of hot tea.\nOne cup


  T=1.0: Every morning before work I drink a cup of coffee. Now, scientists claim


  T=1.0: Every morning before work I drink a cup of Starbucks coffee to increase my cognitive


  T=1.5: Every morning before work I drink a cup of hot cocoa.\nOne cup


  T=1.5: Every morning before work I drink a cup of metastrol yogurt make by Jesse


  T=1.5: Every morning before work I drink a cup of loose–ionized ivy


<div dir="rtl" style="text-align:right"><p>④ בטמפרטורה נמוכה ההמשכים דומים מאוד; בגבוהה הם מתפזרים, ולפעמים יוצאים מוזרים. זו אחת הסיבות שאותה שאלה לצ׳טבוט יכולה לקבל תשובות שונות.</p></div>

<div dir="rtl" style="text-align:right"><p><b>סיימתם את המחברת של שלב B.</b> חזרו ללשונית של אתר הקורס וענו שם על שאלות השלב.</p></div>